In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from itertools import combinations
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import optuna
import warnings
import torch

In [2]:
train_df = pd.read_csv('C:/Users/nihal/Downloads/playground-series-s5e8/train.csv').set_index('id')
test_df = pd.read_csv('C:/Users/nihal/Downloads/playground-series-s5e8/test.csv').set_index('id')
test_df['y'] = -1
submission_df = pd.read_csv('C:/Users/nihal/Downloads/playground-series-s5e8/sample_submission.csv').set_index('id')

In [3]:
print(f'Training data shape: {train_df.shape}')
print(f'Testing data shape: {test_df.shape}')

Training data shape: (750000, 17)
Testing data shape: (250000, 17)


In [4]:
# Display sample of each dataset
print("Sample of Training Data:")
display(train_df.head())

print("\nSample of Test Data:")
display(test_df.head())

Sample of Training Data:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
id,,,,,,,,,,,,,,,,,
0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1



Sample of Test Data:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
id,,,,,,,,,,,,,,,,,
750000,32,blue-collar,married,secondary,no,1397,yes,no,unknown,21,may,224,1,-1,0,unknown,-1
750001,44,management,married,tertiary,no,23,yes,no,cellular,3,apr,586,2,-1,0,unknown,-1
750002,36,self-employed,married,primary,no,46,yes,yes,cellular,13,may,111,2,-1,0,unknown,-1
750003,58,blue-collar,married,secondary,no,-1380,yes,yes,unknown,29,may,125,1,-1,0,unknown,-1
750004,28,technician,single,secondary,no,1950,yes,no,cellular,22,jul,181,1,-1,0,unknown,-1


In [5]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [6]:
# --- Shape, dtypes, and target split ---
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

assert "y" in train_df.columns, "Target y missing from training data."
target = train_df["y"].astype(int)
features = train_df.drop(columns=["y"])

# basic info
display(features.head())
print("\nTarget distribution:")
print(target.value_counts(normalize=True).rename("proportion"))

# class imbalance ratio (neg/pos) for XGB scale_pos_weight
pos = int(target.sum())
neg = int((1 - target).sum())
scale_pos_weight = neg / max(pos, 1)
print(f"\nClass counts -> pos: {pos}, neg: {neg}, scale_pos_weight: {scale_pos_weight:.3f}")

Train shape: (750000, 17)
Test shape : (250000, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
id,,,,,,,,,,,,,,,,
0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown
1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown
2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown
3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown
4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown



Target distribution:
y
0    0.879349
1    0.120651
Name: proportion, dtype: float64

Class counts -> pos: 90488, neg: 659512, scale_pos_weight: 7.288


# Split columns by dtype

In [7]:
# Detect types
numeric_cols = features.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = features.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric columns   :", numeric_cols)
print("Categorical cols  :", categorical_cols)

Numeric columns   : ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical cols  : ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


# Preprocessing: impute + OHE (sparse-friendly)

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from scipy import sparse

# Separate test features
X_test_raw = test_df.drop(columns=["y"])  # 'y' in test is placeholder -1

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# Fit on full training features only (no test leakage)
X_train_enc = preprocess.fit_transform(features)
X_test_enc  = preprocess.transform(X_test_raw)

print("Encoded train shape:", X_train_enc.shape, 
      "| sparse:", sparse.issparse(X_train_enc))
print("Encoded test  shape:", X_test_enc.shape)


Encoded train shape: (750000, 51) | sparse: True
Encoded test  shape: (250000, 51)


# Quick hold-out split for fast feedback metrics

In [10]:
from sklearn.model_selection import StratifiedKFold, train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(
    X_train_enc, target, test_size=0.2, random_state=42, stratify=target
)
print("Holdout shapes ->", X_tr.shape, X_va.shape)

Holdout shapes -> (600000, 51) (150000, 51)


# Baseline XGBoost (reasonable defaults) + full metric suite

In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_recall_curve, average_precision_score

xgb_base = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_base.fit(X_tr, y_tr)
proba_va = xgb_base.predict_proba(X_va)[:, 1]
pred_va  = (proba_va >= 0.5).astype(int)

auc_va   = roc_auc_score(y_va, proba_va)
ap_va    = average_precision_score(y_va, proba_va)  # PR-AUC
acc_va   = accuracy_score(y_va, pred_va)
f1_va    = f1_score(y_va, pred_va)

print(f"Baseline (holdout) -> AUC: {auc_va:.4f} | PR-AUC: {ap_va:.4f} | ACC: {acc_va:.4f} | F1: {f1_va:.4f}")

Baseline (holdout) -> AUC: 0.9669 | PR-AUC: 0.7985 | ACC: 0.8882 | F1: 0.6692


# 5-fold Stratified CV (AUC primary) for a trustworthy estimate

In [12]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

def cv_eval(model, X, y, folds=5, seed=42):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    aucs, accs, f1s, aps = [], [], [], []
    for tr_idx, va_idx in skf.split(X, y):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y.iloc[tr_idx], y.iloc[va_idx]

        model.fit(Xtr, ytr)
        p = model.predict_proba(Xva)[:,1]
        yhat = (p >= 0.5).astype(int)

        aucs.append(roc_auc_score(yva, p))
        aps.append(average_precision_score(yva, p))
        accs.append(accuracy_score(yva, yhat))
        f1s.append(f1_score(yva, yhat))
    return {
        "AUC_mean": np.mean(aucs), "AUC_std": np.std(aucs),
        "PR_mean": np.mean(aps),   "PR_std": np.std(aps),
        "ACC_mean": np.mean(accs), "ACC_std": np.std(accs),
        "F1_mean": np.mean(f1s),   "F1_std": np.std(f1s),
    }

cv_scores = cv_eval(xgb_base, X_train_enc, target, folds=5, seed=42)
cv_scores

{'AUC_mean': np.float64(0.9666466657946741),
 'AUC_std': np.float64(0.0005183125569569888),
 'PR_mean': np.float64(0.7980047099503642),
 'PR_std': np.float64(0.0021163436051918846),
 'ACC_mean': np.float64(0.8868639999999999),
 'ACC_std': np.float64(0.0004850402045191802),
 'F1_mean': np.float64(0.665950905124301),
 'F1_std': np.float64(0.0011311126733815294)}

# Optuna hyperparameter tuning for XGBoost (optimize ROC AUC)

In [13]:
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.2, 3.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        **params
    )
    scores = cv_eval(model, X_train_enc, target, folds=5, seed=42)
    return scores["AUC_mean"]

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best AUC:", study.best_value)
print("Best params:", study.best_params)
best_params = study.best_params

[I 2025-11-11 21:03:01,837] A new study created in memory with name: no-name-c04bfcbd-8da7-4b84-94ad-a83787f9292e


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2025-11-11 21:04:13,177] Trial 0 finished with value: 0.967093016047011 and parameters: {'n_estimators': 700, 'learning_rate': 0.13125830316209655, 'max_depth': 8, 'min_child_weight': 12.374511199743695, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_lambda': 0.23406777359270511, 'reg_alpha': 0.8661761457749352, 'gamma': 3.005575058716044}. Best is trial 0 with value: 0.967093016047011.
[I 2025-11-11 21:06:30,511] Trial 1 finished with value: 0.966218809035731 and parameters: {'n_estimators': 967, 'learning_rate': 0.010573268083515799, 'max_depth': 9, 'min_child_weight': 16.816410175208013, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_lambda': 0.32864757839415243, 'reg_alpha': 0.3042422429595377, 'gamma': 2.6237821581611893}. Best is trial 0 with value: 0.967093016047011.
[I 2025-11-11 21:07:40,039] Trial 2 finished with value: 0.9661559595211061 and parameters: {'n_estimators': 745, 'learning_rate': 0.022004527434741072

# Train best model on full training data + full metrics on holdout

In [14]:
xgb_best = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    **best_params
)

xgb_best.fit(X_tr, y_tr)
proba_va = xgb_best.predict_proba(X_va)[:, 1]
pred_va  = (proba_va >= 0.5).astype(int)

auc_va   = roc_auc_score(y_va, proba_va)
ap_va    = average_precision_score(y_va, proba_va)
acc_va   = accuracy_score(y_va, pred_va)
f1_va    = f1_score(y_va, pred_va)
print(f"Optimized (holdout) -> AUC: {auc_va:.4f} | PR-AUC: {ap_va:.4f} | ACC: {acc_va:.4f} | F1: {f1_va:.4f}")

Optimized (holdout) -> AUC: 0.9680 | PR-AUC: 0.8074 | ACC: 0.8984 | F1: 0.6864


# Confusion matrix & classification report (holdout)

In [15]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_va, pred_va)
print("Confusion matrix (threshold=0.5):\n", cm)
print("\nClassification report:\n", classification_report(y_va, pred_va, digits=4))

Confusion matrix (threshold=0.5):
 [[118067  13835]
 [  1412  16686]]

Classification report:
               precision    recall  f1-score   support

           0     0.9882    0.8951    0.9393    131902
           1     0.5467    0.9220    0.6864     18098

    accuracy                         0.8984    150000
   macro avg     0.7674    0.9085    0.8129    150000
weighted avg     0.9349    0.8984    0.9088    150000



# Train on full training data and create Kaggle submission

In [16]:
# Fit on ALL encoded training data
xgb_best.fit(X_train_enc, target)
test_proba = xgb_best.predict_proba(X_test_enc)[:, 1]

# Create submission
final_submission = submission_df.copy()
final_submission["y"] = test_proba
final_submission.to_csv("xgb_optuna_submission.csv")
final_submission.head()

,y
id,
750000,0.009107
750001,0.320439
750002,0.000438
750003,0.000043
750004,0.073158


In [17]:
# (Optional) Feature importance inspection (gain)
import numpy as np

# WARNING: after OHE, feature names are buried in the ColumnTransformer. This recovers them:
oh = preprocess.named_transformers_["cat"]["oh"]
cat_feature_names = oh.get_feature_names_out(categorical_cols)
num_feature_names = np.array(numeric_cols)

# Build aligned names list
# ColumnTransformer order: num block then cat block (as defined above)
feature_names = np.concatenate([num_feature_names, cat_feature_names])

# Get importances from XGB (trained on full data above)
booster = xgb_best.get_booster()
score_items = booster.get_score(importance_type="gain")  # dict: 'f0': value, ...
# Map 'f{i}' -> feature name
imp = []
for k, v in score_items.items():
    idx = int(k[1:])
    if idx < len(feature_names):
        imp.append((feature_names[idx], v))
imp_sorted = sorted(imp, key=lambda x: x[1], reverse=True)[:25]
pd.DataFrame(imp_sorted, columns=["feature", "gain"]).head(25)


,feature,gain
0,poutcome_success,926.323364
1,contact_unknown,531.959412
2,month_mar,505.631317
3,duration,336.471802
4,month_oct,173.879303
5,housing_no,170.356308
6,month_sep,126.890198
7,poutcome_unknown,120.463860
8,housing_yes,104.418327
9,month_jan,96.819466
